# 02-14c nnU-Net PHE with peri-ICH ring prior

This notebook branches from `02_14b` after observing that `CT + ICH probability + distance` reduced false positives but increased false negatives.

Main changes:

- keep the ICH teacher frozen and reuse existing PHE-SICH teacher predictions when available;
- create a peri-ICH/ring prior: high outside the hematoma, zero inside the hematoma;
- default experiment: `CT + ICH probability + peri-ICH prior`;
- save PHE probability maps and run a probability-threshold sweep to recover recall;
- keep raw, postprocessed, detection, and threshold-sweep outputs separate from `14b`.


In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import time
import gc

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
NNUNET_REPO = PROJECT_ROOT / 'nnUNet-master'

PHE_ROOT = PROJECT_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT'
PHE_IMAGE_DIR = PHE_ROOT / 'set'
PHE_MASK_DIR = PHE_ROOT / 'label'
SPLIT_CSV = PROJECT_ROOT / 'outputs_02_10_pese_guided_3dff_refined_pseudo_iph_phe_25d_segmentation' / 'manifests' / '3dff_iph_phe_patient_split.csv'

ICH_TEACHER_CONFIGURATION = '3d_fullres'
ICH_TEACHER_TRAINER = 'nnUNetTrainer_250epochs'
ICH_TEACHER_PLANS = 'nnUNetPlans'
ICH_TEACHER_FOLD = 0
ICH_TEACHER_CHECKPOINT = 'checkpoint_best.pth'

ICH_TEACHER_CANDIDATES = [
    {
        'source': '02_13b',
        'root': PROJECT_ROOT / 'outputs_02_13b_nnunet_ich_teacher_external_improved',
        'dataset_name': 'Dataset123_ExternalICHTeacherImproved',
    },
    {
        'source': '02_13',
        'root': PROJECT_ROOT / 'outputs_02_13_nnunet_ich_teacher_external',
        'dataset_name': 'Dataset113_ExternalICHTeacher',
    },
]

def resolve_teacher_candidate():
    checked = []
    for candidate in ICH_TEACHER_CANDIDATES:
        model_dir = (
            candidate['root'] / 'nnunet_workdir' / 'nnUNet_results' / candidate['dataset_name'] /
            f'{ICH_TEACHER_TRAINER}__{ICH_TEACHER_PLANS}__{ICH_TEACHER_CONFIGURATION}'
        )
        ckpt_path = model_dir / f'fold_{ICH_TEACHER_FOLD}' / ICH_TEACHER_CHECKPOINT
        checked.append(str(ckpt_path))
        if ckpt_path.exists():
            return candidate['source'], candidate['root'], candidate['dataset_name'], model_dir, ckpt_path
    raise FileNotFoundError('No teacher checkpoint found. Checked:\n' + '\n'.join(checked))

(
    ICH_TEACHER_SOURCE,
    ICH_TEACHER_ROOT,
    ICH_TEACHER_DATASET_NAME,
    ICH_TEACHER_MODEL_DIR,
    ICH_TEACHER_CHECKPOINT_PATH,
) = resolve_teacher_candidate()

EXPERIMENTS = {
    'phe_probability_prior': {
        'dataset_id': 128,
        'prior_channels': ['ich_probability'],
        'description': 'CT + soft ICH probability map',
    },
    'phe_peri_ring_prior': {
        'dataset_id': 129,
        'prior_channels': ['ich_peri_prior'],
        'description': 'CT + peri-ICH soft ring prior',
    },
    'phe_prob_peri_ring_prior': {
        'dataset_id': 130,
        'prior_channels': ['ich_probability', 'ich_peri_prior'],
        'description': 'CT + ICH probability + peri-ICH soft ring prior',
    },
    'phe_prob_peri_ring_roi_prior': {
        'dataset_id': 131,
        'prior_channels': ['ich_probability', 'ich_peri_prior', 'ich_ring_roi'],
        'description': 'CT + ICH probability + peri-ICH soft ring + ring ROI',
    },
}

EXPERIMENT_ID = 'phe_prob_peri_ring_prior'
assert EXPERIMENT_ID in EXPERIMENTS, f'Unknown EXPERIMENT_ID: {EXPERIMENT_ID}'
EXPERIMENT = EXPERIMENTS[EXPERIMENT_ID]
PRIOR_CHANNELS = list(EXPERIMENT['prior_channels'])

OUTPUT_BASE = PROJECT_ROOT / 'outputs_02_14c_nnunet_phe_ich_teacher_peri_ring_prior'
OUTPUT_ROOT = OUTPUT_BASE / EXPERIMENT_ID
TEACHER_OUTPUT_ROOT = OUTPUT_BASE / 'teacher_outputs' / 'phe_sich_ich_prior'
REUSE_TEACHER_OUTPUT_ROOTS = [
    TEACHER_OUTPUT_ROOT,
    PROJECT_ROOT / 'outputs_02_14b_nnunet_phe_ich_teacher_soft_spatial_prior' / 'teacher_outputs' / 'phe_sich_ich_prior',
]
def resolve_teacher_output_root(candidates):
    for root in candidates:
        pred_dir = root / 'nnunet_predictions'
        if pred_dir.exists() and len(list(pred_dir.glob('*.npz'))) > 0:
            return root
    return TEACHER_OUTPUT_ROOT

ICH_SOURCE_OUTPUT_ROOT = resolve_teacher_output_root(REUSE_TEACHER_OUTPUT_ROOTS)
NNUNET_BASE = OUTPUT_ROOT / 'nnunet_workdir'
NNUNET_RAW = NNUNET_BASE / 'nnUNet_raw'
NNUNET_PREPROCESSED = NNUNET_BASE / 'nnUNet_preprocessed'
# Keep results short on Windows. The default nested output path can exceed MAX_PATH during training.
NNUNET_RESULTS = PROJECT_ROOT / 'o14c_results' / EXPERIMENT_ID

DATASET_ID = int(EXPERIMENT['dataset_id'])
DATASET_NAME = f'Dataset{DATASET_ID:03d}_PHESICH_PHE_{EXPERIMENT_ID}'
DATASET_DIR = NNUNET_RAW / DATASET_NAME

ICH_INPUT_DIR = ICH_SOURCE_OUTPUT_ROOT / 'inputs'
ICH_RAW_PRED_DIR = ICH_SOURCE_OUTPUT_ROOT / 'nnunet_predictions'
ICH_PRIOR_PROB_DIR = TEACHER_OUTPUT_ROOT / 'probability_maps'
ICH_PRIOR_MASK_DIR = TEACHER_OUTPUT_ROOT / 'binary_masks'
ICH_DISTANCE_PRIOR_DIR = TEACHER_OUTPUT_ROOT / 'distance_maps'
ICH_DILATED_ROI_DIR = TEACHER_OUTPUT_ROOT / 'dilated_roi'
ICH_PERI_PRIOR_DIR = TEACHER_OUTPUT_ROOT / 'peri_ich_maps'
ICH_RING_ROI_DIR = TEACHER_OUTPUT_ROOT / 'peri_ring_roi'
PRIOR_QC_DIR = TEACHER_OUTPUT_ROOT / 'visualizations'

for p in [
    OUTPUT_BASE, OUTPUT_ROOT, TEACHER_OUTPUT_ROOT,
    NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS,
    ICH_INPUT_DIR, ICH_RAW_PRED_DIR, ICH_PRIOR_PROB_DIR,
    ICH_PRIOR_MASK_DIR, ICH_DISTANCE_PRIOR_DIR, ICH_DILATED_ROI_DIR, ICH_PERI_PRIOR_DIR, ICH_RING_ROI_DIR, PRIOR_QC_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ['nnUNet_n_proc_DA'] = '0'
os.environ['nnUNet_compile'] = 'False'

if str(NNUNET_REPO) not in sys.path:
    sys.path.insert(0, str(NNUNET_REPO))

assert NNUNET_REPO.exists(), f'Missing nnU-Net repo: {NNUNET_REPO}'
assert PHE_IMAGE_DIR.exists(), f'Missing PHE image dir: {PHE_IMAGE_DIR}'
assert PHE_MASK_DIR.exists(), f'Missing PHE mask dir: {PHE_MASK_DIR}'
assert SPLIT_CSV.exists(), f'Missing split CSV: {SPLIT_CSV}'
assert ICH_TEACHER_CHECKPOINT_PATH.exists(), f'Missing ICH teacher checkpoint: {ICH_TEACHER_CHECKPOINT_PATH}'

print('Project root          :', PROJECT_ROOT)
print('Experiment            :', EXPERIMENT_ID, '-', EXPERIMENT['description'])
print('Prior channels        :', PRIOR_CHANNELS)
print('nnU-Net repo          :', NNUNET_REPO)
print('PHE image dir         :', PHE_IMAGE_DIR)
print('PHE mask dir          :', PHE_MASK_DIR)
print('Split CSV            :', SPLIT_CSV)
print('ICH teacher source   :', ICH_TEACHER_SOURCE)
print('ICH teacher model    :', ICH_TEACHER_MODEL_DIR)
print('ICH teacher ckpt     :', ICH_TEACHER_CHECKPOINT_PATH)
print('Output root          :', OUTPUT_ROOT)
print('Teacher output root  :', TEACHER_OUTPUT_ROOT)
print('Teacher source root  :', ICH_SOURCE_OUTPUT_ROOT)
print('Dataset              :', DATASET_NAME)
print('nnUNet_raw           :', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed  :', os.environ['nnUNet_preprocessed'])
print('nnUNet_results       :', os.environ['nnUNet_results'])


Project root          : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao
Experiment            : phe_prob_peri_ring_prior - CT + ICH probability + peri-ICH soft ring prior
Prior channels        : ['ich_probability', 'ich_peri_prior']
nnU-Net repo          : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\nnUNet-master
PHE image dir         : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-IDS\SubdatasetA_NIFIT\NIFIT\set
PHE mask dir          : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-IDS\SubdatasetA_NIFIT\NIFIT\label
Split CSV            : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_10_pese_guided_3dff_refined_pseudo_iph_phe_25d_segmentation\manifests\3dff_iph_phe_patient_split.csv
ICH teacher source   : 02_13b
ICH teacher model    : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_results\Dataset123_ExternalICHTeacherImproved\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres
ICH teacher ckpt     : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet

## 0. Dependency check

In [2]:
AUTO_INSTALL_MISSING_DEPS = False
INSTALL_NNUNET = False

def ensure_import(import_name, pip_name=None):
    import importlib
    import subprocess
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        if not AUTO_INSTALL_MISSING_DEPS:
            raise
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)

if INSTALL_NNUNET:
    import subprocess
    cmd = [sys.executable, '-m', 'pip', 'install', '-e', str(NNUNET_REPO)]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

ensure_import('SimpleITK')
ensure_import('scipy')
ensure_import('nnunetv2')

import SimpleITK as sitk
from scipy import ndimage as ndi
import nnunetv2
print('SimpleITK OK')
print('scipy OK')
print('nnunetv2 import OK:', nnunetv2.__file__)


SimpleITK OK
scipy OK
nnunetv2 import OK: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\nnUNet-master\nnunetv2\__init__.py


## 1. Inspect split PHE-SICH

Dung lai split tu `02_10`/`02_12`: train 99, val 10, test 11.

In [3]:
split_df = pd.read_csv(SPLIT_CSV, dtype={'scan_id': str, 'patient_id': str})
split_df.columns = [str(c).strip() for c in split_df.columns]

def make_nnunet_case_id(value, img_path=None):
    value = '' if value is None or pd.isna(value) else str(value).strip()
    if not value and img_path is not None:
        name = Path(str(img_path)).name
        value = name.replace('.nii.gz', '').replace('.nii', '')
    if value.lower().startswith('phe_'):
        return value
    if value.isdigit():
        return f'phe_{int(value):04d}'
    safe = ''.join(ch if ch.isalnum() else '_' for ch in value).strip('_')
    return f'phe_{safe}' if not safe.lower().startswith('phe_') else safe

if 'patient_id' in split_df.columns:
    split_df['case_id'] = [make_nnunet_case_id(v, p) for v, p in zip(split_df['patient_id'], split_df['img_path'])]
elif 'scan_id' in split_df.columns:
    split_df['case_id'] = [make_nnunet_case_id(v, p) for v, p in zip(split_df['scan_id'], split_df['img_path'])]
else:
    split_df['case_id'] = [make_nnunet_case_id(None, p) for p in split_df['img_path']]

required_cols = {'case_id', 'split', 'img_path', 'phe_mask_path'}
missing_cols = required_cols - set(split_df.columns)
assert not missing_cols, f'Missing columns in split CSV: {missing_cols}'
assert split_df['case_id'].is_unique, 'case_id must be unique.'

split_df['img_exists'] = split_df['img_path'].map(lambda x: Path(x).exists())
split_df['mask_exists'] = split_df['phe_mask_path'].map(lambda x: Path(x).exists())
assert split_df['img_exists'].all(), 'Some image paths are missing.'
assert split_df['mask_exists'].all(), 'Some PHE mask paths are missing.'

display(split_df.groupby('split').agg(
    cases=('case_id', 'count'),
    slices=('n_slices', 'sum'),
    phe_positive_slices=('phe_positive_slices', 'sum'),
    total_phe_ml=('phe_volume_ml', 'sum'),
))
display(split_df[['case_id', 'split', 'img_path', 'phe_mask_path']].head())
split_df.to_csv(OUTPUT_ROOT / '02_14_phe_sich_split_manifest.csv', index=False)
print('Total cases:', len(split_df))


,cases,slices,phe_positive_slices,total_phe_ml
split,,,,
test,11,329,62,31.534920
train,99,2983,560,355.115621
val,10,289,61,34.999343


,case_id,split,img_path,phe_mask_path
0,phe_0001,train,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
1,phe_0002,test,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
2,phe_0004,train,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
3,phe_0005,train,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...
4,phe_0006,train,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\PHE-SICH-CT-...


Total cases: 120


## 2. Predict/export ICH priors on PHE-SICH

This cell reuses existing teacher predictions from `02_14b` when available, then exports priors for `14c`:

```text
ICH probability map
ICH binary mask
distance-to-ICH proximity map
peri-ICH soft ring prior = high near hematoma, but zero inside hematoma
peri-ICH ring ROI
```

The ICH teacher is not retrained here.


In [4]:
RUN_ICH_PRIOR_PREDICT = True
OVERWRITE_ICH_INPUTS = True
OVERWRITE_ICH_PRIORS = False
ICH_DISABLE_TTA = False
TEACHER_PROB_THRESHOLD = 0.5
DISTANCE_SCALE_MM = 10.0
DILATED_ROI_RADIUS_MM = 20.0
PERI_DISTANCE_SCALE_MM = 10.0
PERI_RING_OUTER_RADIUS_MM = 30.0

USING_REUSED_TEACHER_OUTPUT = Path(ICH_SOURCE_OUTPUT_ROOT).resolve() != Path(TEACHER_OUTPUT_ROOT).resolve()
if USING_REUSED_TEACHER_OUTPUT:
    print('Reusing teacher raw predictions from:', ICH_SOURCE_OUTPUT_ROOT)
else:
    if OVERWRITE_ICH_INPUTS and ICH_INPUT_DIR.exists():
        shutil.rmtree(ICH_INPUT_DIR)
    ICH_INPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [ICH_PRIOR_PROB_DIR, ICH_PRIOR_MASK_DIR, ICH_DISTANCE_PRIOR_DIR, ICH_DILATED_ROI_DIR, ICH_PERI_PRIOR_DIR, ICH_RING_ROI_DIR]:
    p.mkdir(parents=True, exist_ok=True)
ICH_RAW_PRED_DIR.mkdir(parents=True, exist_ok=True)

input_records = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    src = Path(row.img_path)
    dst = ICH_INPUT_DIR / f'{case_id}_0000.nii.gz'
    if not USING_REUSED_TEACHER_OUTPUT and (OVERWRITE_ICH_INPUTS or not dst.exists()):
        shutil.copy2(src, dst)
    input_records.append({'case_id': case_id, 'split': row.split, 'ich_teacher_input': str(dst)})

ich_input_df = pd.DataFrame(input_records)
ich_input_df.to_csv(TEACHER_OUTPUT_ROOT / '02_14c_ich_teacher_input_manifest.csv', index=False)

expected_raw_npz = [ICH_RAW_PRED_DIR / f'{case_id}.npz' for case_id in split_df['case_id'].astype(str)]
expected_raw_seg = [ICH_RAW_PRED_DIR / f'{case_id}.nii.gz' for case_id in split_df['case_id'].astype(str)]
missing_raw_npz = [p for p in expected_raw_npz if not p.exists()]
missing_raw_seg = [p for p in expected_raw_seg if not p.exists()]
missing_raw = missing_raw_npz + missing_raw_seg
print('ICH teacher inputs:', len(list(ICH_INPUT_DIR.glob('*.nii.gz'))) if ICH_INPUT_DIR.exists() else 0)
print('Existing raw teacher predictions:', (2 * len(split_df)) - len(missing_raw), '/', 2 * len(split_df))

if RUN_ICH_PRIOR_PREDICT and (OVERWRITE_ICH_PRIORS or missing_raw):
    if USING_REUSED_TEACHER_OUTPUT:
        raise FileNotFoundError(f'Reused teacher output is incomplete: {len(missing_raw)} missing files in {ICH_RAW_PRED_DIR}')
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('ICH teacher predict device:', device)
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not ICH_DISABLE_TTA,
        perform_everything_on_device=(device.type == 'cuda'),
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(ICH_TEACHER_MODEL_DIR),
        use_folds=(ICH_TEACHER_FOLD,),
        checkpoint_name=ICH_TEACHER_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(ICH_INPUT_DIR),
        str(ICH_RAW_PRED_DIR),
        save_probabilities=True,
        overwrite=OVERWRITE_ICH_PRIORS or bool(missing_raw_npz),
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
elif RUN_ICH_PRIOR_PREDICT:
    print('All raw ICH teacher predictions already exist. Set OVERWRITE_ICH_PRIORS=True to regenerate.')
else:
    print('Skipped ICH teacher inference. Set RUN_ICH_PRIOR_PREDICT=True to run.')

def load_nnunet_probability(npz_path: Path):
    with np.load(npz_path) as data:
        key = 'probabilities' if 'probabilities' in data.files else data.files[0]
        arr = np.asarray(data[key])
    if arr.ndim == 4:
        return arr[1] if arr.shape[0] > 1 else arr[0]
    if arr.ndim == 3:
        return arr
    raise ValueError(f'Unexpected probability shape {arr.shape}: {npz_path}')

def write_array_like_reference(arr, reference_path: Path, out_path: Path, dtype=np.float32):
    ref = sitk.ReadImage(str(reference_path))
    out = sitk.GetImageFromArray(np.asarray(arr).astype(dtype))
    if out.GetSize() != ref.GetSize():
        raise ValueError(f'Size mismatch for {out_path}: {out.GetSize()} vs ref {ref.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(out_path))

def make_distance_prior(binary_ich, spacing_xyz, scale_mm=10.0):
    binary_ich = binary_ich.astype(bool)
    if not binary_ich.any():
        return np.zeros(binary_ich.shape, dtype=np.float32)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return np.exp(-dist_mm / float(scale_mm)).astype(np.float32)

def make_dilated_roi(binary_ich, spacing_xyz, radius_mm=20.0):
    binary_ich = binary_ich.astype(bool)
    if not binary_ich.any():
        return np.zeros(binary_ich.shape, dtype=np.uint8)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return (dist_mm <= float(radius_mm)).astype(np.uint8)

def make_peri_ich_prior(binary_ich, spacing_xyz, scale_mm=10.0, outer_radius_mm=30.0):
    binary_ich = binary_ich.astype(bool)
    if not binary_ich.any():
        return np.zeros(binary_ich.shape, dtype=np.float32)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    prior = np.exp(-dist_mm / float(scale_mm)).astype(np.float32)
    prior[binary_ich] = 0.0
    if outer_radius_mm is not None:
        prior[dist_mm > float(outer_radius_mm)] = 0.0
    return np.clip(prior, 0.0, 1.0).astype(np.float32)

def make_peri_ring_roi(binary_ich, spacing_xyz, outer_radius_mm=30.0):
    binary_ich = binary_ich.astype(bool)
    if not binary_ich.any():
        return np.zeros(binary_ich.shape, dtype=np.uint8)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return ((dist_mm <= float(outer_radius_mm)) & (~binary_ich)).astype(np.uint8)

prior_rows = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    npz_path = ICH_RAW_PRED_DIR / f'{case_id}.npz'
    raw_seg_path = ICH_RAW_PRED_DIR / f'{case_id}.nii.gz'
    if npz_path.exists():
        prob = load_nnunet_probability(npz_path)
        source = 'probability_npz'
    elif raw_seg_path.exists():
        prob = (sitk.GetArrayFromImage(sitk.ReadImage(str(raw_seg_path))) > 0).astype(np.float32)
        source = 'fallback_binary_prediction'
    else:
        raise FileNotFoundError(f'Missing raw teacher output for {case_id}: {npz_path} or {raw_seg_path}')

    ref_path = raw_seg_path if raw_seg_path.exists() else Path(row.img_path)
    ref_img = sitk.ReadImage(str(ref_path))
    spacing_xyz = np.array(ref_img.GetSpacing(), dtype=float)
    prob = np.clip(prob.astype(np.float32), 0.0, 1.0)
    binary = (prob >= TEACHER_PROB_THRESHOLD).astype(np.uint8)
    distance_prior = make_distance_prior(binary, spacing_xyz, scale_mm=DISTANCE_SCALE_MM)
    dilated_roi = make_dilated_roi(binary, spacing_xyz, radius_mm=DILATED_ROI_RADIUS_MM)
    peri_prior = make_peri_ich_prior(binary, spacing_xyz, scale_mm=PERI_DISTANCE_SCALE_MM, outer_radius_mm=PERI_RING_OUTER_RADIUS_MM)
    ring_roi = make_peri_ring_roi(binary, spacing_xyz, outer_radius_mm=PERI_RING_OUTER_RADIUS_MM)

    prob_path = ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz'
    binary_path = ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'
    distance_path = ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz'
    dilated_path = ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz'
    peri_path = ICH_PERI_PRIOR_DIR / f'{case_id}.nii.gz'
    ring_path = ICH_RING_ROI_DIR / f'{case_id}.nii.gz'
    write_array_like_reference(prob, ref_path, prob_path, dtype=np.float32)
    write_array_like_reference(binary, ref_path, binary_path, dtype=np.uint8)
    write_array_like_reference(distance_prior, ref_path, distance_path, dtype=np.float32)
    write_array_like_reference(dilated_roi, ref_path, dilated_path, dtype=np.uint8)
    write_array_like_reference(peri_prior, ref_path, peri_path, dtype=np.float32)
    write_array_like_reference(ring_roi, ref_path, ring_path, dtype=np.uint8)

    prior_rows.append({
        'case_id': case_id,
        'split': row.split,
        'source': source,
        'probability_path': str(prob_path),
        'binary_path': str(binary_path),
        'distance_path': str(distance_path),
        'dilated_roi_path': str(dilated_path),
        'peri_prior_path': str(peri_path),
        'ring_roi_path': str(ring_path),
        'prob_min': float(prob.min()),
        'prob_max': float(prob.max()),
        'prob_mean': float(prob.mean()),
        'binary_voxels': int(binary.sum()),
        'dilated_voxels': int(dilated_roi.sum()),
        'peri_prior_mean': float(peri_prior.mean()),
        'peri_prior_voxels_gt0': int((peri_prior > 0).sum()),
        'ring_roi_voxels': int(ring_roi.sum()),
    })

prior_df = pd.DataFrame(prior_rows)
prior_manifest = TEACHER_OUTPUT_ROOT / '02_14c_ich_prior_manifest.csv'
prior_df.to_csv(prior_manifest, index=False)
display(prior_df.groupby('split').agg(
    cases=('case_id', 'count'),
    mean_prob=('prob_mean', 'mean'),
    binary_voxels=('binary_voxels', 'sum'),
    peri_prior_voxels=('peri_prior_voxels_gt0', 'sum'),
    ring_roi_voxels=('ring_roi_voxels', 'sum'),
).reset_index())
print('Saved prior manifest:', prior_manifest)
print('Probability prior dir:', ICH_PRIOR_PROB_DIR)
print('Binary prior dir     :', ICH_PRIOR_MASK_DIR)
print('Peri-ICH prior dir   :', ICH_PERI_PRIOR_DIR)
print('Peri ring ROI dir    :', ICH_RING_ROI_DIR)


Reusing teacher raw predictions from: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14b_nnunet_phe_ich_teacher_soft_spatial_prior\teacher_outputs\phe_sich_ich_prior
ICH teacher inputs: 120
Existing raw teacher predictions: 240 / 240
All raw ICH teacher predictions already exist. Set OVERWRITE_ICH_PRIORS=True to regenerate.


,split,cases,mean_prob,binary_voxels,peri_prior_voxels,ring_roi_voxels
0,test,11,0.001571,135119,6022494,6022494
1,train,99,0.002204,1612207,73693927,73693927
2,val,10,0.002144,159428,6604722,6604722


Saved prior manifest: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14c_nnunet_phe_ich_teacher_peri_ring_prior\teacher_outputs\phe_sich_ich_prior\02_14c_ich_prior_manifest.csv
Probability prior dir: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14c_nnunet_phe_ich_teacher_peri_ring_prior\teacher_outputs\phe_sich_ich_prior\probability_maps
Binary prior dir     : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14c_nnunet_phe_ich_teacher_peri_ring_prior\teacher_outputs\phe_sich_ich_prior\binary_masks
Peri-ICH prior dir   : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14c_nnunet_phe_ich_teacher_peri_ring_prior\teacher_outputs\phe_sich_ich_prior\peri_ich_maps
Peri ring ROI dir    : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14c_nnunet_phe_ich_teacher_peri_ring_prior\teacher_outputs\phe_sich_ich_prior\peri_ring_roi


## 3. Quick inspect ICH/peri prior maps

In [5]:
INSPECT_PRIORS = True
MAX_INSPECT = 8

if INSPECT_PRIORS:
    inspect_rows = []
    for row in split_df.head(MAX_INSPECT).itertuples(index=False):
        case_id = str(row.case_id)
        paths = {
            'probability': ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz',
            'binary': ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz',
            'distance': ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz',
            'dilated_roi': ICH_DILATED_ROI_DIR / f'{case_id}.nii.gz',
            'peri_prior': ICH_PERI_PRIOR_DIR / f'{case_id}.nii.gz',
            'ring_roi': ICH_RING_ROI_DIR / f'{case_id}.nii.gz',
        }
        record = {'case_id': case_id, 'split': row.split}
        for name, path in paths.items():
            if path.exists():
                arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
                record[f'{name}_shape_zyx'] = tuple(arr.shape)
                record[f'{name}_min'] = float(np.min(arr))
                record[f'{name}_max'] = float(np.max(arr))
                record[f'{name}_voxels_gt0'] = int((arr > 0).sum())
            else:
                record[f'{name}_status'] = 'missing'
        inspect_rows.append(record)
    display(pd.DataFrame(inspect_rows))
else:
    print('Skipped prior inspection.')


,case_id,split,probability_shape_zyx,probability_min,probability_max,probability_voxels_gt0,binary_shape_zyx,binary_min,binary_max,binary_voxels_gt0,...,dilated_roi_max,dilated_roi_voxels_gt0,peri_prior_shape_zyx,peri_prior_min,peri_prior_max,peri_prior_voxels_gt0,ring_roi_shape_zyx,ring_roi_min,ring_roi_max,ring_roi_voxels_gt0
0,phe_0001,train,"(32, 512, 512)",1.210620e-28,0.999998,8388608,"(32, 512, 512)",0.0,1.0,8442,...,1.0,133603,"(32, 512, 512)",0.0,0.950487,282277,"(32, 512, 512)",0.0,1.0,282277
1,phe_0002,test,"(32, 512, 512)",5.161602e-22,1.000000,8388608,"(32, 512, 512)",0.0,1.0,1326,...,1.0,145351,"(32, 512, 512)",0.0,0.952345,373316,"(32, 512, 512)",0.0,1.0,373316
2,phe_0004,train,"(32, 512, 512)",3.487262e-19,0.999987,8388608,"(32, 512, 512)",0.0,1.0,13358,...,1.0,294547,"(32, 512, 512)",0.0,0.957941,639255,"(32, 512, 512)",0.0,1.0,639255
3,phe_0005,train,"(32, 512, 512)",4.290686e-17,1.000000,8388608,"(32, 512, 512)",0.0,1.0,5478,...,1.0,344830,"(32, 512, 512)",0.0,0.957941,881693,"(32, 512, 512)",0.0,1.0,881693
4,phe_0006,train,"(32, 512, 512)",8.905035e-19,0.999968,8388608,"(32, 512, 512)",0.0,1.0,38352,...,1.0,482487,"(32, 512, 512)",0.0,0.957941,1033953,"(32, 512, 512)",0.0,1.0,1033953
5,phe_0008,train,"(32, 512, 512)",5.831280e-16,0.999996,8388608,"(32, 512, 512)",0.0,1.0,16954,...,1.0,249433,"(32, 512, 512)",0.0,0.952345,571447,"(32, 512, 512)",0.0,1.0,571447
6,phe_0009,train,"(32, 512, 512)",1.853177e-15,1.000000,8388608,"(32, 512, 512)",0.0,1.0,3644,...,1.0,159769,"(32, 512, 512)",0.0,0.952345,409335,"(32, 512, 512)",0.0,1.0,409335
7,phe_0011,train,"(30, 512, 512)",1.392982e-17,1.000000,7864320,"(30, 512, 512)",0.0,1.0,4735,...,1.0,220084,"(30, 512, 512)",0.0,0.952345,526853,"(30, 512, 512)",0.0,1.0,526853


## 3b. QC visualization for PHE-SICH peri priors

This checks whether the new peri-ICH prior highlights tissue around hematoma instead of the hematoma core itself.


In [6]:
RUN_PRIOR_QC_FIGURES = True
N_PRIOR_QC_PER_BUCKET = 4

CT_WINDOWS = {
    'brain': (0, 80),
    'blood': (-50, 150),
    'wide': (-100, 300),
}

def window_ct(arr, low=-50, high=150):
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.clip(arr, low, high)
    return (arr - low) / max(high - low, 1e-6)

def largest_slice(mask):
    if not mask.any():
        return mask.shape[0] // 2
    areas = mask.reshape(mask.shape[0], -1).sum(axis=1)
    return int(np.argmax(areas))

def select_prior_qc_cases(prior_df, n_each=4):
    buckets = []
    buckets.append(prior_df.sort_values('binary_voxels').head(n_each).assign(qc_reason='small_prior'))
    buckets.append(prior_df.sort_values('binary_voxels', ascending=False).head(n_each).assign(qc_reason='large_prior'))
    test_df = prior_df[prior_df['split'] == 'test']
    if len(test_df):
        buckets.append(test_df.head(n_each).assign(qc_reason='test_cases'))
    out = pd.concat(buckets, ignore_index=True)
    return out.drop_duplicates('case_id', keep='first')

if RUN_PRIOR_QC_FIGURES:
    import matplotlib.pyplot as plt
    qc_df = select_prior_qc_cases(prior_df, N_PRIOR_QC_PER_BUCKET)
    qc_df.to_csv(TEACHER_OUTPUT_ROOT / '02_14c_prior_qc_selected_cases.csv', index=False)
    for row in qc_df.itertuples(index=False):
        case_id = str(row.case_id)
        split_row = split_df.loc[split_df['case_id'].astype(str) == case_id].iloc[0]
        ct = sitk.GetArrayFromImage(sitk.ReadImage(str(split_row.img_path)))
        phe = sitk.GetArrayFromImage(sitk.ReadImage(str(split_row.phe_mask_path))) > 0
        prob = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_PRIOR_PROB_DIR / f'{case_id}.nii.gz')))
        binary = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_PRIOR_MASK_DIR / f'{case_id}.nii.gz'))) > 0
        distance = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_DISTANCE_PRIOR_DIR / f'{case_id}.nii.gz')))
        peri = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_PERI_PRIOR_DIR / f'{case_id}.nii.gz')))
        ring = sitk.GetArrayFromImage(sitk.ReadImage(str(ICH_RING_ROI_DIR / f'{case_id}.nii.gz'))) > 0
        z = largest_slice(phe | binary | ring)
        ct_blood = window_ct(ct[z], *CT_WINDOWS['blood'])
        ct_brain = window_ct(ct[z], *CT_WINDOWS['brain'])
        fig, axes = plt.subplots(1, 5, figsize=(18, 4))
        axes[0].imshow(ct_brain, cmap='gray')
        axes[0].set_title(f'{case_id}\n{row.qc_reason}\nCT brain')
        axes[1].imshow(ct_blood, cmap='gray')
        axes[1].imshow(np.ma.masked_where(~phe[z], phe[z]), cmap='Greens', alpha=0.55)
        axes[1].set_title('PHE GT')
        axes[2].imshow(ct_blood, cmap='gray')
        axes[2].imshow(prob[z], cmap='magma', alpha=0.60, vmin=0, vmax=1)
        axes[2].set_title('ICH probability')
        axes[3].imshow(peri[z], cmap='viridis', vmin=0, vmax=1)
        axes[3].set_title('peri-ICH prior')
        axes[4].imshow(ct_blood, cmap='gray')
        axes[4].imshow(np.ma.masked_where(~ring[z], ring[z]), cmap='Blues', alpha=0.25)
        axes[4].imshow(np.ma.masked_where(~phe[z], phe[z]), cmap='Greens', alpha=0.45)
        axes[4].imshow(np.ma.masked_where(~binary[z], binary[z]), cmap='Reds', alpha=0.45)
        axes[4].set_title('PHE + ICH + ring')
        for ax in axes:
            ax.axis('off')
        fig.tight_layout()
        out = PRIOR_QC_DIR / f'{case_id}_{row.qc_reason}_prior_qc.png'
        fig.savefig(out, dpi=160, bbox_inches='tight')
        plt.close(fig)
    print('Saved prior QC figures:', PRIOR_QC_DIR)
else:
    print('Skipped prior QC figures.')


Saved prior QC figures: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_14c_nnunet_phe_ich_teacher_peri_ring_prior\teacher_outputs\phe_sich_ich_prior\visualizations


## 4. Convert PHE-SICH to selectable multi-channel nnU-Net raw dataset

Default `14c` experiment:

```text
channel 0000 = CT
channel 0001 = ICH probability
channel 0002 = peri-ICH soft ring prior
label = manual PHE mask
```

The peri prior is zero inside ICH and high just outside ICH, matching the assumption that PHE surrounds hematoma.


In [7]:
OVERWRITE_RAW_DATASET = True

PRIOR_CHANNEL_PATHS = {
    'ich_binary': ICH_PRIOR_MASK_DIR,
    'ich_probability': ICH_PRIOR_PROB_DIR,
    'ich_distance': ICH_DISTANCE_PRIOR_DIR,
    'ich_dilated_roi': ICH_DILATED_ROI_DIR,
    'ich_peri_prior': ICH_PERI_PRIOR_DIR,
    'ich_ring_roi': ICH_RING_ROI_DIR,
}
PRIOR_CHANNEL_DISPLAY_NAMES = {
    'ich_binary': 'ICH_binary_prior',
    'ich_probability': 'ICH_probability_prior',
    'ich_distance': 'ICH_distance_prior',
    'ich_dilated_roi': 'ICH_dilated_roi_prior',
    'ich_peri_prior': 'ICH_peri_ring_prior',
    'ich_ring_roi': 'ICH_ring_roi_prior',
}

def write_mask_like_reference(src_path: Path, reference_path: Path, dst_path: Path):
    ref = sitk.ReadImage(str(reference_path))
    seg = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(seg)
    out = sitk.GetImageFromArray((arr > 0).astype(np.uint8))
    if ref.GetSize() != out.GetSize():
        raise ValueError(f'Size mismatch: reference {reference_path} {ref.GetSize()} vs {src_path} {out.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(dst_path))

def write_prior_like_reference(src_path: Path, reference_path: Path, dst_path: Path, prior_name: str):
    ref = sitk.ReadImage(str(reference_path))
    src = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(src)
    if prior_name in {'ich_binary', 'ich_dilated_roi', 'ich_ring_roi'}:
        arr = (arr > 0).astype(np.uint8)
        dtype = np.uint8
    else:
        arr = np.clip(arr.astype(np.float32), 0.0, 1.0)
        dtype = np.float32
    out = sitk.GetImageFromArray(arr.astype(dtype))
    if ref.GetSize() != out.GetSize():
        raise ValueError(f'Size mismatch: reference {reference_path} {ref.GetSize()} vs {src_path} {out.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(dst_path))
    return float(arr.min()), float(arr.max()), int((arr > 0).sum())

if DATASET_DIR.exists() and OVERWRITE_RAW_DATASET:
    shutil.rmtree(DATASET_DIR)

imagesTr = DATASET_DIR / 'imagesTr'
labelsTr = DATASET_DIR / 'labelsTr'
imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
for p in [imagesTr, labelsTr, imagesTs, labelsTs]:
    p.mkdir(parents=True, exist_ok=True)

records = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    img_src = Path(row.img_path)
    phe_src = Path(row.phe_mask_path)

    if row.split == 'test':
        image_dir = imagesTs
        label_dir = labelsTs
    else:
        image_dir = imagesTr
        label_dir = labelsTr

    img_dst = image_dir / f'{case_id}_0000.nii.gz'
    seg_dst = label_dir / f'{case_id}.nii.gz'
    shutil.copy2(img_src, img_dst)

    channel_record = {}
    for channel_offset, prior_name in enumerate(PRIOR_CHANNELS, start=1):
        prior_dir = PRIOR_CHANNEL_PATHS[prior_name]
        prior_src = prior_dir / f'{case_id}.nii.gz'
        assert prior_src.exists(), f'Missing {prior_name} for {case_id}: {prior_src}'
        prior_dst = image_dir / f'{case_id}_{channel_offset:04d}.nii.gz'
        prior_min, prior_max, prior_voxels = write_prior_like_reference(prior_src, img_dst, prior_dst, prior_name)
        channel_record[f'{prior_name}_channel'] = str(prior_dst)
        channel_record[f'{prior_name}_min'] = prior_min
        channel_record[f'{prior_name}_max'] = prior_max
        channel_record[f'{prior_name}_voxels_gt0'] = prior_voxels

    write_mask_like_reference(phe_src, img_dst, seg_dst)
    records.append({
        'case_id': case_id,
        'split': row.split,
        'ct_channel': str(img_dst),
        'phe_label': str(seg_dst),
        'source_image': str(img_src),
        'source_phe_mask': str(phe_src),
        **channel_record,
    })

channel_names = {'0': 'CT'}
for channel_offset, prior_name in enumerate(PRIOR_CHANNELS, start=1):
    channel_names[str(channel_offset)] = PRIOR_CHANNEL_DISPLAY_NAMES[prior_name]

dataset_json = {
    'channel_names': channel_names,
    'labels': {'background': 0, 'PHE': 1},
    'numTraining': int((split_df['split'] != 'test').sum()),
    'file_ending': '.nii.gz',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}
with open(DATASET_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_json, f, indent=2)

conversion_df = pd.DataFrame(records)
conversion_manifest = OUTPUT_ROOT / f'02_14c_{EXPERIMENT_ID}_conversion_manifest.csv'
conversion_df.to_csv(conversion_manifest, index=False)

print('Experiment:', EXPERIMENT_ID)
print('Prior channels:', PRIOR_CHANNELS)
print('Dataset dir:', DATASET_DIR)
print('imagesTr channel files:', len(list(imagesTr.glob('*.nii.gz'))))
print('labelsTr:', len(list(labelsTr.glob('*.nii.gz'))))
print('imagesTs channel files:', len(list(imagesTs.glob('*.nii.gz'))))
print('labelsTs:', len(list(labelsTs.glob('*.nii.gz'))))
print('dataset.json:', DATASET_DIR / 'dataset.json')
print('conversion manifest:', conversion_manifest)
display(conversion_df.groupby('split').size().rename('cases').reset_index())
display(pd.DataFrame([dataset_json['channel_names']]).T.rename(columns={0: 'channel_name'}))


OSError: [Errno 28] No space left on device

## 5. Helper call entrypoints

In [ ]:
def call_entrypoint(entrypoint_func, argv):
    old_argv = sys.argv[:]
    sys.argv = list(argv)
    print('>>>', ' '.join(map(str, sys.argv)))
    t0 = time.time()
    try:
        return entrypoint_func()
    finally:
        sys.argv = old_argv
        print(f'Done in {(time.time() - t0) / 60:.1f} min')


## 6. Plan and preprocess selected Dataset

In [ ]:
CONFIGURATION = '3d_fullres'
RUN_PLAN_PREPROCESS = True

if RUN_PLAN_PREPROCESS:
    from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry
    old_n_proc_da = os.environ.get('nnUNet_n_proc_DA')
    os.environ['nnUNet_n_proc_DA'] = '1'
    try:
        call_entrypoint(plan_and_preprocess_entry, [
            'nnUNetv2_plan_and_preprocess',
            '-d', str(DATASET_ID),
            '-c', CONFIGURATION,
            '--verify_dataset_integrity',
            '-npfp', '2',
            '-np', '1',
        ])
    finally:
        os.environ['nnUNet_n_proc_DA'] = old_n_proc_da if old_n_proc_da is not None else '0'
else:
    print('Skipped. Set RUN_PLAN_PREPROCESS=True to run.')


## 7. Write manual fold-0 split

In [ ]:
preprocessed_dataset_dir = NNUNET_PREPROCESSED / DATASET_NAME
split_json_path = preprocessed_dataset_dir / 'splits_final.json'

train_ids = split_df.loc[split_df['split'] == 'train', 'case_id'].astype(str).tolist()
val_ids = split_df.loc[split_df['split'] == 'val', 'case_id'].astype(str).tolist()
test_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()

manual_splits = [{'train': train_ids, 'val': val_ids}]

if preprocessed_dataset_dir.exists():
    with open(split_json_path, 'w', encoding='utf-8') as f:
        json.dump(manual_splits, f, indent=2)
    print('Wrote:', split_json_path)
    print('train:', len(train_ids), 'val:', len(val_ids), 'test:', len(test_ids))
else:
    print('Preprocessed folder not found yet:', preprocessed_dataset_dir)
    print('Run plan/preprocess first, then rerun this cell.')


## 8. Train PHE model with selected ICH prior channels

This trains a new PHE model for the current `EXPERIMENT_ID`. The ICH teacher remains frozen.


In [ ]:
RUN_TRAIN = True
CONFIGURATION = globals().get('CONFIGURATION', '3d_fullres')
FOLD = 0
TRAINER = 'nnUNetTrainer_250epochs'
PLANS = 'nnUNetPlans'
EXPORT_VAL_NPZ = True
CONTINUE_TRAINING = True

MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FOLD_DIR.mkdir(parents=True, exist_ok=True)

EFFECTIVE_CONTINUE_TRAINING = CONTINUE_TRAINING
if CONTINUE_TRAINING:
    available_checkpoints = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth'))
    if not available_checkpoints:
        print('CONTINUE_TRAINING=True but no checkpoint exists yet; starting a fresh training run.')
        EFFECTIVE_CONTINUE_TRAINING = False
    else:
        print('Continuing from available checkpoints:', available_checkpoints)

if RUN_TRAIN:
    import torch
    from nnunetv2.run.run_training import run_training
    os.environ['nnUNet_n_proc_DA'] = '0'
    os.environ['nnUNet_compile'] = 'False'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Training device:', device)
    print('Model dir:', MODEL_DIR)
    print('Fold dir :', FOLD_DIR)
    print('nnUNet_n_proc_DA:', os.environ.get('nnUNet_n_proc_DA'))
    print('nnUNet_compile:', os.environ.get('nnUNet_compile'))
    print('continue_training:', EFFECTIVE_CONTINUE_TRAINING)
    # Windows/Jupyter guard: some nnU-Net builds try to copy dataset_fingerprint.json
    # before recreating the trainer output folder. Ensure the destination parent exists.
    import shutil as _shutil
    if not hasattr(_shutil, '_codex_original_copyfile'):
        _shutil._codex_original_copyfile = _shutil.copyfile

    def _copyfile_with_parent(src, dst, *args, **kwargs):
        if str(src).endswith('dataset_fingerprint.json'):
            Path(dst).parent.mkdir(parents=True, exist_ok=True)
        return _shutil._codex_original_copyfile(src, dst, *args, **kwargs)

    _shutil.copyfile = _copyfile_with_parent
    try:
        run_training(
            str(DATASET_ID),
            CONFIGURATION,
            str(FOLD),
            trainer_class_name=TRAINER,
            plans_identifier=PLANS,
            export_validation_probabilities=EXPORT_VAL_NPZ,
            continue_training=EFFECTIVE_CONTINUE_TRAINING,
            device=device,
        )
    finally:
        _shutil.copyfile = _shutil._codex_original_copyfile
else:
    print('Skipped. Set RUN_TRAIN=True to train.')


## 9. Predict test set and evaluate PHE

In [ ]:
RUN_PHE_PREDICT = True
RUN_PHE_EVALUATE = True
DISABLE_TTA = False
CHECKPOINT = 'auto'

imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'

def resolve_checkpoint_name(checkpoint_setting='auto'):
    if checkpoint_setting != 'auto':
        ckpt = FOLD_DIR / checkpoint_setting
        if not ckpt.exists():
            available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
            raise FileNotFoundError(f'Missing checkpoint: {ckpt}. Available: {available}')
        return checkpoint_setting
    for name in ['checkpoint_best.pth', 'checkpoint_final.pth', 'checkpoint_latest.pth']:
        if (FOLD_DIR / name).exists():
            return name
    available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
    raise FileNotFoundError(f'No checkpoint found in {FOLD_DIR}. Available: {available}')

RESOLVED_CHECKPOINT = resolve_checkpoint_name(CHECKPOINT)
PRED_DIR = OUTPUT_ROOT / f'predictions_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
SUMMARY_JSON = PRED_DIR / 'summary.json'

print('Model dir        :', MODEL_DIR)
print('Fold dir         :', FOLD_DIR)
print('Using checkpoint :', RESOLVED_CHECKPOINT)

if RUN_PHE_PREDICT:
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('PHE predict device:', device)
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not DISABLE_TTA,
        perform_everything_on_device=(device.type == 'cuda'),
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(MODEL_DIR),
        use_folds=(FOLD,),
        checkpoint_name=RESOLVED_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(imagesTs),
        str(PRED_DIR),
        save_probabilities=True,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
else:
    print('Skipped prediction. Set RUN_PHE_PREDICT=True to run inference.')

if RUN_PHE_EVALUATE:
    from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
    plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
    dataset_json_file = DATASET_DIR / 'dataset.json'
    call_entrypoint(evaluate_folder_entry_point, [
        'nnUNetv2_evaluate_folder',
        str(labelsTs),
        str(PRED_DIR),
        '-djfile', str(dataset_json_file),
        '-pfile', str(plans_file),
        '-o', str(SUMMARY_JSON),
        '-np', '2',
    ])
else:
    print('Skipped evaluation. Set RUN_PHE_EVALUATE=True after prediction.')

print('Prediction dir:', PRED_DIR)
print('Summary json  :', SUMMARY_JSON)


## 9a. PHE probability threshold sweep

`14b` suggested that the prior reduced FP but increased FN. This cell evaluates lower PHE probability thresholds to see whether recall and Dice improve without retraining.


In [ ]:
RUN_PHE_PROB_THRESHOLD_SWEEP = True
PHE_PROB_THRESHOLDS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
PHE_THRESHOLD_SWEEP_ROOT = OUTPUT_ROOT / f"phe_probability_threshold_sweep_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace('.pth', '')}"

def write_threshold_mask(prob_arr, reference_path: Path, out_path: Path, threshold: float):
    ref = sitk.ReadImage(str(reference_path))
    mask = (np.asarray(prob_arr, dtype=np.float32) >= float(threshold)).astype(np.uint8)
    out = sitk.GetImageFromArray(mask)
    if out.GetSize() != ref.GetSize():
        raise ValueError(f'Size mismatch for {out_path}: {out.GetSize()} vs ref {ref.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(out_path))
    return int(mask.sum())

if RUN_PHE_PROB_THRESHOLD_SWEEP:
    from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
    plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
    dataset_json_file = DATASET_DIR / 'dataset.json'
    threshold_rows = []
    missing_prob = []
    for threshold in PHE_PROB_THRESHOLDS:
        threshold_tag = f'thr_{threshold:.2f}'.replace('.', '_')
        threshold_dir = PHE_THRESHOLD_SWEEP_ROOT / threshold_tag
        threshold_dir.mkdir(parents=True, exist_ok=True)
        voxel_counts = []
        for case_id in split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str):
            npz_path = PRED_DIR / f'{case_id}.npz'
            ref_path = PRED_DIR / f'{case_id}.nii.gz'
            if not npz_path.exists():
                missing_prob.append(str(npz_path))
                continue
            prob = load_nnunet_probability(npz_path)
            voxel_counts.append(write_threshold_mask(prob, ref_path, threshold_dir / f'{case_id}.nii.gz', threshold))
        if missing_prob:
            print('Missing probability files; run prediction with save_probabilities=True first.')
            break
        summary_path = threshold_dir / 'summary.json'
        call_entrypoint(evaluate_folder_entry_point, [
            'nnUNetv2_evaluate_folder',
            str(labelsTs),
            str(threshold_dir),
            '-djfile', str(dataset_json_file),
            '-pfile', str(plans_file),
            '-o', str(summary_path),
            '-np', '2',
        ])
        with open(summary_path, 'r', encoding='utf-8') as f:
            summary = json.load(f)
        fg = summary.get('foreground_mean', {})
        threshold_rows.append({
            'experiment_id': EXPERIMENT_ID,
            'threshold': float(threshold),
            'prediction_dir': str(threshold_dir),
            'summary_json': str(summary_path),
            'mean_pred_voxels': float(np.mean(voxel_counts)) if voxel_counts else np.nan,
            **{f'foreground_{k}': v for k, v in fg.items()},
        })
    phe_threshold_sweep_df = pd.DataFrame(threshold_rows)
    threshold_csv = OUTPUT_ROOT / f'02_14c_{EXPERIMENT_ID}_phe_probability_threshold_sweep.csv'
    phe_threshold_sweep_df.to_csv(threshold_csv, index=False)
    print('Wrote:', threshold_csv)
    display(phe_threshold_sweep_df)
else:
    print('Skipped PHE probability threshold sweep.')


## 9b. ICH/peri-ring guided post-processing

This optional post-processing keeps PHE components that overlap the peri-ICH ring ROI. It is mainly diagnostic: if it improves Dice, false positives far from hematoma are a problem.


In [ ]:
RUN_POSTPROCESS = True
POSTPROCESS_MIN_SIZE_VOXELS = 50
POSTPROCESS_ROI_RADIUS_MM = 25.0

POSTPROCESS_PRED_DIR = OUTPUT_ROOT / f'postprocessed_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
POSTPROCESS_SUMMARY_JSON = POSTPROCESS_PRED_DIR / 'summary.json'

def make_ich_roi(binary_ich, spacing_xyz, radius_mm):
    binary_ich = binary_ich.astype(bool)
    if not binary_ich.any():
        return np.ones(binary_ich.shape, dtype=bool)
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist_mm = ndi.distance_transform_edt(~binary_ich, sampling=spacing_zyx)
    return dist_mm <= float(radius_mm)

def postprocess_phe_with_ich(phe_pred, ich_prior, spacing_xyz, min_size=50, radius_mm=25.0):
    phe_pred = phe_pred.astype(bool)
    ich_prior = ich_prior.astype(bool)
    ich_roi = make_ich_roi(ich_prior, spacing_xyz, radius_mm)
    labeled, num = ndi.label(phe_pred)
    output = np.zeros_like(phe_pred, dtype=np.uint8)
    kept = 0
    removed_small = 0
    removed_far = 0
    for comp_id in range(1, num + 1):
        comp = labeled == comp_id
        comp_size = int(comp.sum())
        if comp_size < int(min_size):
            removed_small += 1
            continue
        if np.logical_and(comp, ich_roi).sum() == 0:
            removed_far += 1
            continue
        output[comp] = 1
        kept += 1
    return output, {'components': int(num), 'kept': kept, 'removed_small': removed_small, 'removed_far': removed_far}

def write_mask_array_like_reference(mask_arr, reference_path: Path, out_path: Path):
    ref = sitk.ReadImage(str(reference_path))
    out = sitk.GetImageFromArray(mask_arr.astype(np.uint8))
    if out.GetSize() != ref.GetSize():
        raise ValueError(f'Size mismatch for {out_path}: {out.GetSize()} vs ref {ref.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(out_path))

if RUN_POSTPROCESS:
    POSTPROCESS_PRED_DIR.mkdir(parents=True, exist_ok=True)
    pp_rows = []
    for case_id in split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str):
        raw_pred_path = PRED_DIR / f'{case_id}.nii.gz'
        ich_prior_path = ICH_RING_ROI_DIR / f'{case_id}.nii.gz'
        out_path = POSTPROCESS_PRED_DIR / f'{case_id}.nii.gz'
        if not raw_pred_path.exists():
            print('Missing raw PHE prediction:', raw_pred_path)
            continue
        raw_img = sitk.ReadImage(str(raw_pred_path))
        raw_pred = sitk.GetArrayFromImage(raw_img) > 0
        ich_prior = sitk.GetArrayFromImage(sitk.ReadImage(str(ich_prior_path))) > 0
        spacing_xyz = np.array(raw_img.GetSpacing(), dtype=float)
        pp_pred, stats = postprocess_phe_with_ich(
            raw_pred,
            ich_prior,
            spacing_xyz,
            min_size=POSTPROCESS_MIN_SIZE_VOXELS,
            radius_mm=POSTPROCESS_ROI_RADIUS_MM,
        )
        write_mask_array_like_reference(pp_pred, raw_pred_path, out_path)
        pp_rows.append({
            'case_id': case_id,
            'raw_voxels': int(raw_pred.sum()),
            'post_voxels': int(pp_pred.sum()),
            **stats,
        })
    postprocess_df = pd.DataFrame(pp_rows)
    postprocess_manifest = OUTPUT_ROOT / f'02_14c_{EXPERIMENT_ID}_postprocess_manifest.csv'
    postprocess_df.to_csv(postprocess_manifest, index=False)
    print('Saved postprocessed predictions:', POSTPROCESS_PRED_DIR)
    print('Saved postprocess manifest:', postprocess_manifest)
    display(postprocess_df)

    if RUN_PHE_EVALUATE and len(postprocess_df):
        from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
        plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
        dataset_json_file = DATASET_DIR / 'dataset.json'
        call_entrypoint(evaluate_folder_entry_point, [
            'nnUNetv2_evaluate_folder',
            str(labelsTs),
            str(POSTPROCESS_PRED_DIR),
            '-djfile', str(dataset_json_file),
            '-pfile', str(plans_file),
            '-o', str(POSTPROCESS_SUMMARY_JSON),
            '-np', '2',
        ])
else:
    print('Skipped ICH-guided post-processing.')


## 10. Export compact metrics CSV

In [ ]:
def strip_nii_gz_name(path_like):
    name = Path(str(path_like)).name
    return name[:-7] if name.endswith('.nii.gz') else Path(name).stem

def export_summary(summary_json: Path, source_name: str):
    if not summary_json.exists():
        print('No summary yet:', summary_json)
        return None, None
    with open(summary_json, 'r', encoding='utf-8') as f:
        summary = json.load(f)

    rows = []
    for item in summary.get('metric_per_case', []):
        metrics = item.get('metrics', {}).get('1', {})
        rows.append({
            'experiment_id': EXPERIMENT_ID,
            'source': source_name,
            'case_id': strip_nii_gz_name(item.get('prediction_file', '')),
            'prediction_file': item.get('prediction_file', ''),
            'reference_file': item.get('reference_file', ''),
            **metrics,
        })
    per_case_df = pd.DataFrame(rows)

    summary_rows = []
    numeric_cols = [c for c in per_case_df.columns if c not in {'experiment_id', 'source', 'case_id', 'prediction_file', 'reference_file'}]
    for col in numeric_cols:
        vals = pd.to_numeric(per_case_df[col], errors='coerce').dropna()
        if len(vals) == 0:
            continue
        summary_rows.append({
            'experiment_id': EXPERIMENT_ID,
            'source': source_name,
            'label': 'PHE',
            'metric': col,
            'mean': float(vals.mean()),
            'median': float(vals.median()),
            'std': float(vals.std(ddof=0)),
            'n': int(len(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    return per_case_df, summary_df

raw_per_case_df, raw_summary_df = export_summary(SUMMARY_JSON, 'raw')
post_per_case_df, post_summary_df = export_summary(globals().get('POSTPROCESS_SUMMARY_JSON', Path('__missing__')), 'postprocessed')

per_case_frames = [df for df in [raw_per_case_df, post_per_case_df] if df is not None]
summary_frames = [df for df in [raw_summary_df, post_summary_df] if df is not None]

if per_case_frames:
    metrics_df = pd.concat(per_case_frames, ignore_index=True)
    metrics_csv = OUTPUT_ROOT / f'02_14c_{EXPERIMENT_ID}_metrics_per_case_raw_vs_post.csv'
    metrics_df.to_csv(metrics_csv, index=False)
    print('Wrote:', metrics_csv)
else:
    print('No per-case metrics exported.')

if summary_frames:
    summary_df = pd.concat(summary_frames, ignore_index=True)
    summary_csv = OUTPUT_ROOT / f'02_14c_{EXPERIMENT_ID}_metrics_summary_raw_vs_post.csv'
    summary_df.to_csv(summary_csv, index=False)
    print('Wrote:', summary_csv)
    display(summary_df.pivot_table(index='metric', columns='source', values='mean'))
else:
    print('No metric summary exported. Run train -> predict -> evaluate first.')

experiment_record = {
    'experiment_id': EXPERIMENT_ID,
    'dataset_id': DATASET_ID,
    'dataset_name': DATASET_NAME,
    'prior_channels': '|'.join(PRIOR_CHANNELS),
    'teacher_source': ICH_TEACHER_SOURCE,
    'teacher_checkpoint': str(ICH_TEACHER_CHECKPOINT_PATH),
    'output_root': str(OUTPUT_ROOT),
}
experiment_summary_csv = OUTPUT_BASE / '02_14c_experiment_registry.csv'
existing = pd.read_csv(experiment_summary_csv) if experiment_summary_csv.exists() else pd.DataFrame()
registry_df = pd.concat([existing, pd.DataFrame([experiment_record])], ignore_index=True)
registry_df = registry_df.drop_duplicates(['experiment_id', 'dataset_id'], keep='last')
registry_df.to_csv(experiment_summary_csv, index=False)
print('Updated experiment registry:', experiment_summary_csv)


## 11. PHE detection / early detection analysis

This section converts the PHE segmentation output into a patient-level detection result. Because the current split manifest does not include onset-to-CT time, `early` is treated as a low-volume/subtle-PHE proxy rather than a true time-based early window.


In [ ]:
RUN_PHE_DETECTION_ANALYSIS = True

# GT positive threshold should usually stay at 0: any manual PHE voxel means PHE is present.
GT_POSITIVE_VOLUME_THRESHOLD_ML = 0.0

# Main operating point. The threshold sweep below helps decide whether this should change.
PRED_POSITIVE_VOLUME_THRESHOLD_ML = 0.5
DETECTION_THRESHOLD_SWEEP_ML = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0]

# Proxy for early/subtle PHE when onset-time metadata is unavailable.
SMALL_PHE_VOLUME_CUTOFF_ML = 5.0


def load_binary_mask(path: Path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img) > 0
    return img, arr


def mask_volume_ml(mask, spacing_xyz):
    voxel_ml = float(np.prod(np.asarray(spacing_xyz, dtype=float)) / 1000.0)
    return float(mask.sum() * voxel_ml)


def safe_div(num, den):
    return np.nan if den == 0 else float(num / den)


def detection_counts(gt_positive, pred_positive):
    gt_positive = np.asarray(gt_positive, dtype=bool)
    pred_positive = np.asarray(pred_positive, dtype=bool)
    tp = int(np.logical_and(gt_positive, pred_positive).sum())
    fp = int(np.logical_and(~gt_positive, pred_positive).sum())
    fn = int(np.logical_and(gt_positive, ~pred_positive).sum())
    tn = int(np.logical_and(~gt_positive, ~pred_positive).sum())
    return tp, fp, fn, tn


def detection_metric_row(df, source, cohort, pred_threshold_ml):
    tp, fp, fn, tn = detection_counts(df['gt_positive'], df['pred_positive'])
    sensitivity = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    precision = safe_div(tp, tp + fp)
    recall = sensitivity
    npv = safe_div(tn, tn + fn)
    f1 = np.nan if precision != precision or recall != recall or (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    accuracy = safe_div(tp + tn, tp + fp + fn + tn)
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    return {
        'experiment_id': EXPERIMENT_ID,
        'source': source,
        'cohort': cohort,
        'pred_threshold_ml': float(pred_threshold_ml),
        'gt_threshold_ml': float(GT_POSITIVE_VOLUME_THRESHOLD_ML),
        'small_phe_cutoff_ml': float(SMALL_PHE_VOLUME_CUTOFF_ML),
        'n': int(len(df)),
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'recall': recall,
        'npv': npv,
        'f1': f1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'gt_positive_rate': float(pd.to_numeric(df['gt_positive'], errors='coerce').mean()) if len(df) else np.nan,
        'pred_positive_rate': float(pd.to_numeric(df['pred_positive'], errors='coerce').mean()) if len(df) else np.nan,
        'mean_gt_volume_ml': float(df['gt_volume_ml'].mean()) if len(df) else np.nan,
        'mean_pred_volume_ml': float(df['pred_volume_ml'].mean()) if len(df) else np.nan,
    }


def available_detection_sources():
    sources = []
    if 'PRED_DIR' in globals():
        sources.append(('raw', Path(PRED_DIR)))
    if 'POSTPROCESS_PRED_DIR' in globals() and Path(POSTPROCESS_PRED_DIR).exists():
        sources.append(('postprocessed', Path(POSTPROCESS_PRED_DIR)))
    return sources


if RUN_PHE_DETECTION_ANALYSIS:
    if 'labelsTs' not in globals():
        raise RuntimeError('labelsTs is not defined. Run dataset conversion/prediction cells before this detection analysis.')

    detection_sources = available_detection_sources()
    if not detection_sources:
        print('No prediction folders found. Run PHE prediction first, then rerun this cell.')
    else:
        case_rows = []
        test_case_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()
        for source_name, pred_dir in detection_sources:
            for case_id in test_case_ids:
                gt_path = labelsTs / f'{case_id}.nii.gz'
                pred_path = pred_dir / f'{case_id}.nii.gz'
                row = {
                    'experiment_id': EXPERIMENT_ID,
                    'source': source_name,
                    'case_id': case_id,
                    'gt_path': str(gt_path),
                    'pred_path': str(pred_path),
                    'status': 'ok',
                }
                if not gt_path.exists():
                    row['status'] = 'missing_gt'
                    case_rows.append(row)
                    continue
                if not pred_path.exists():
                    row['status'] = 'missing_prediction'
                    case_rows.append(row)
                    continue

                gt_img, gt_mask = load_binary_mask(gt_path)
                _, pred_mask = load_binary_mask(pred_path)
                if gt_mask.shape != pred_mask.shape:
                    row['status'] = f'shape_mismatch_gt_{gt_mask.shape}_pred_{pred_mask.shape}'
                    case_rows.append(row)
                    continue

                spacing_xyz = np.array(gt_img.GetSpacing(), dtype=float)
                gt_volume_ml = mask_volume_ml(gt_mask, spacing_xyz)
                pred_volume_ml = mask_volume_ml(pred_mask, spacing_xyz)
                gt_positive = gt_volume_ml > GT_POSITIVE_VOLUME_THRESHOLD_ML
                pred_positive = pred_volume_ml >= PRED_POSITIVE_VOLUME_THRESHOLD_ML
                row.update({
                    'gt_volume_ml': gt_volume_ml,
                    'pred_volume_ml': pred_volume_ml,
                    'gt_voxels': int(gt_mask.sum()),
                    'pred_voxels': int(pred_mask.sum()),
                    'gt_positive': bool(gt_positive),
                    'pred_positive': bool(pred_positive),
                    'small_phe_proxy': bool(gt_positive and gt_volume_ml <= SMALL_PHE_VOLUME_CUTOFF_ML),
                    'volume_error_ml': float(pred_volume_ml - gt_volume_ml),
                    'abs_volume_error_ml': float(abs(pred_volume_ml - gt_volume_ml)),
                })
                case_rows.append(row)

        detection_case_df = pd.DataFrame(case_rows)
        detection_cases_csv = OUTPUT_ROOT / f'02_14c_{EXPERIMENT_ID}_phe_detection_per_case.csv'
        detection_case_df.to_csv(detection_cases_csv, index=False)
        print('Wrote:', detection_cases_csv)

        valid_df = detection_case_df[detection_case_df['status'] == 'ok'].copy()
        summary_rows = []
        threshold_rows = []
        if len(valid_df):
            for source_name, source_df in valid_df.groupby('source'):
                gt_class_counts = source_df['gt_positive'].value_counts(dropna=False).to_dict()
                print(f'GT detection class counts for {source_name}:', gt_class_counts)
                if source_df['gt_positive'].nunique(dropna=True) < 2:
                    print(f'Warning: {source_name} has only one GT detection class; specificity and balanced accuracy may be limited.')
                summary_rows.append(detection_metric_row(source_df, source_name, 'all_test', PRED_POSITIVE_VOLUME_THRESHOLD_ML))

                small_df = source_df[source_df['small_phe_proxy']]
                if len(small_df):
                    summary_rows.append(detection_metric_row(
                        small_df,
                        source_name,
                        f'small_phe_le_{SMALL_PHE_VOLUME_CUTOFF_ML:g}ml',
                        PRED_POSITIVE_VOLUME_THRESHOLD_ML,
                    ))

                large_df = source_df[source_df['gt_positive'] & ~source_df['small_phe_proxy']]
                if len(large_df):
                    summary_rows.append(detection_metric_row(
                        large_df,
                        source_name,
                        f'larger_phe_gt_{SMALL_PHE_VOLUME_CUTOFF_ML:g}ml',
                        PRED_POSITIVE_VOLUME_THRESHOLD_ML,
                    ))

                for threshold_ml in DETECTION_THRESHOLD_SWEEP_ML:
                    sweep_df = source_df.copy()
                    sweep_df['pred_positive'] = sweep_df['pred_volume_ml'] >= float(threshold_ml)
                    threshold_rows.append(detection_metric_row(sweep_df, source_name, 'threshold_sweep_all_test', threshold_ml))

            detection_summary_df = pd.DataFrame(summary_rows)
            detection_threshold_sweep_df = pd.DataFrame(threshold_rows)
            detection_summary_csv = OUTPUT_ROOT / f'02_14c_{EXPERIMENT_ID}_phe_detection_summary.csv'
            detection_threshold_sweep_csv = OUTPUT_ROOT / f'02_14c_{EXPERIMENT_ID}_phe_detection_threshold_sweep.csv'
            detection_summary_df.to_csv(detection_summary_csv, index=False)
            detection_threshold_sweep_df.to_csv(detection_threshold_sweep_csv, index=False)
            print('Wrote:', detection_summary_csv)
            print('Wrote:', detection_threshold_sweep_csv)
            display(detection_summary_df[['source', 'cohort', 'n', 'tp', 'fp', 'fn', 'tn', 'sensitivity', 'specificity', 'precision', 'f1', 'accuracy']])
            display(detection_threshold_sweep_df[['source', 'pred_threshold_ml', 'sensitivity', 'specificity', 'precision', 'f1', 'accuracy']])
            display(valid_df.sort_values(['source', 'gt_volume_ml'])[['source', 'case_id', 'gt_volume_ml', 'pred_volume_ml', 'gt_positive', 'pred_positive', 'small_phe_proxy']])
        else:
            print('No valid prediction/GT pairs found for detection analysis.')
else:
    print('Skipped PHE detection analysis.')


## Notes

- `02_14c` tests whether a peri-ICH/ring prior works better than the direct distance-to-ICH prior from `14b`.
- Default experiment is `phe_prob_peri_ring_prior`: CT + ICH probability + peri-ICH soft ring prior.
- The teacher raw predictions are reused from `14b` when available, so the expensive teacher inference does not need to run again.
- Prediction saves PHE probabilities and section 9a evaluates lower probability thresholds to recover recall.
- Keep comparisons against `02_14` and `02_14b` on the same split.
